# NB15 — Publication figures, tables, and a results bundle
Every number here is read from files written by NB04–NB14. Nothing is re-estimated, except Kaplan-Meier curves split
at the cohort median of M, which are **for display only** (the analysis uses continuous scores, no cut-points).

Outputs
* `16_Publication_Figures/` — Fig1–Fig6 (PNG, PDF, SVG) and supplementary figures copied with stable names.
* `15_Final_Results/` — Table1 cohorts, Table2 Cox results, Table3 top candidates, Supplementary tables (CSV + one Excel file).
* `18_Manuscript/results_digest.md` — key numbers and a methods summary in plain text, to draft the paper from.
* `17_MEDCON_Presentation/slide_outline.md` — slide-by-slide outline with the numbers filled in.
* `15_Final_Results/GBM_results_bundle.zip` — figures, tables and the two text files in one file, ready to share.

In [ ]:
import os, sys, json, shutil, zipfile
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB15 needs full-run results.")
verify_frozen()
FIG = P("figures"); FIN = P("final"); MS = P("manuscript"); MED = P("medcon"); CL = P("clinical"); VAL = P("validation")


def jload(p):
    return json.loads(p.read_text()) if p.exists() else {}


summary = jload(FIN / "results_summary.json")
require(bool(summary), "results_summary.json not found; run NB14 first.")
manifest = jload(P("logs", "run_manifest.json"))
cohorts = [c for c in ["TCGA", "CGGA693", "CGGA325"] if (VAL / f"{c}_scores.csv").exists()]
scores = {c: pd.read_csv(VAL / f"{c}_scores.csv") for c in cohorts}

In [ ]:
# ---- Table 1: cohorts --------------------------------------------------------------------------------------
rows = []
for c, d in scores.items():
    sex = d.sex.astype(str).str.strip().str.lower()
    m = d.get("mgmt", pd.Series(dtype=str)).astype(str).str.strip().str.lower()
    rows.append({"cohort": c, "role": "training" if c == CFG["survival"]["training_cohort"] else "validation",
                 "platform": "RNA-seq (Toil, TCGA)" if c == "TCGA" else "RNA-seq (CGGA, FPKM)",
                 "n": len(d), "deaths": int(d.os_event.sum()), "median_OS_days": float(d.os_time_days.median()),
                 "age_median": float(pd.to_numeric(d.age, errors="coerce").median()),
                 "male_pct": float((sex.str.startswith("m")).mean() * 100),
                 "MGMT_methylated_pct": float((m == "methylated").mean() * 100) if (m != "nan").any() else np.nan})
t1 = pd.DataFrame(rows)
de_sum = pd.read_csv(P("de", "DE_summary_all_cohorts.csv"))
t1.to_csv(FIN / "Table1_survival_cohorts.csv", index=False)
de_sum.to_csv(FIN / "TableS1_DE_cohorts.csv", index=False)
print(t1.round(1).to_string(index=False)); print(de_sum.to_string(index=False))

# ---- Table 2: Cox meta-analysis -----------------------------------------------------------------------------
cm = pd.read_csv(CL / "cox_meta_analysis.csv")
cb = pd.read_csv(CL / "cox_by_cohort.csv")
t2 = cm[cm.model.isin(["m1", "m3"])][["model", "predictor", "n_cohorts", "HR_fixed", "HR_fixed_lo", "HR_fixed_hi", "p_fixed", "HR_random", "p_random", "I2"]]
t2.to_csv(FIN / "Table2_cox_meta_S_M_R.csv", index=False)
cb.to_csv(FIN / "TableS2_cox_by_cohort.csv", index=False)
print(t2.round(3).to_string(index=False))

# ---- Table 3: candidates -------------------------------------------------------------------------------------
cand = pd.read_csv(FIN / "candidate_table_tiered.csv")
keep_cols = [c for c in ["gene", "tier", "meta_logHR", "meta_FDR", "I2", "direction_concordant", "loo_stable", "n_any", "n_functional", "provisional_label"] if c in cand]
t3 = cand[keep_cols].head(30)
t3.to_csv(FIN / "Table3_top_candidates.csv", index=False)
with pd.ExcelWriter(FIN / "All_tables.xlsx") as xw:
    for nm, t in [("Table1", t1), ("Table2", t2), ("Table3", t3), ("S1_DE", de_sum), ("S2_cox_by_cohort", cb)]:
        t.to_excel(xw, sheet_name=nm, index=False)
    for nm, f in [("S3_sensitivity", P("sensitivity", "sensitivity_results.csv", mkdir=False)),
                  ("S4_H2_concordance", P("meta_analysis", "H2_control_type_concordance.csv", mkdir=False)),
                  ("S5_H2_markers", P("meta_analysis", "H2_marker_enrichment.csv", mkdir=False)),
                  ("S6_evidence_map", FIN / "evidence_map.csv")]:
        if f.exists():
            pd.read_csv(f).to_excel(xw, sheet_name=nm, index=False)
print("Tables written to", FIN)

In [ ]:
# ---- Kaplan-Meier (display only; median split of M) ----------------------------------------------------------
def km(t, e):
    order = np.argsort(t); t, e = np.asarray(t)[order], np.asarray(e)[order]
    times, surv, s = [0.0], [1.0], 1.0
    for u in np.unique(t):
        d = e[t == u].sum(); at_risk = (t >= u).sum()
        if d > 0:
            s *= 1 - d / at_risk; times.append(u); surv.append(s)
    return np.array(times), np.array(surv)


fig, axes = plt.subplots(1, len(cohorts), figsize=(4.3 * len(cohorts), 3.8), squeeze=False)
for ax, c in zip(axes[0], cohorts):
    d = scores[c].dropna(subset=["M", "os_time_days", "os_event"])
    hi = d.M >= d.M.median()
    for mask, lab, col in [(~hi, "M low", "#0f9d94"), (hi, "M high", "#1f3a93")]:
        tt, ss = km(d.os_time_days[mask] / 30.44, d.os_event[mask])
        ax.step(tt, ss, where="post", color=col, label=f"{lab} (n={int(mask.sum())})")
    x = cb[(cb.cohort == c) & (cb.model == "m3") & (cb.predictor == "M")]
    hr_txt = f"adjusted HR per SD {x.HR.iloc[0]:.2f} ({x.HR_lo.iloc[0]:.2f}-{x.HR_hi.iloc[0]:.2f})" if len(x) else ""
    ax.set_title(f"{c}\n{hr_txt}", fontsize=9); ax.set_xlabel("months"); ax.set_ylabel("overall survival"); ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Display only: median split of the M score (models use continuous scores)", fontsize=9, y=1.02)
fig.tight_layout(); savefig(fig, "Fig4_KM_M_median_split_display", folder="figures"); plt.close(fig)

# ---- copy main figures with stable names ---------------------------------------------------------------------
copies = [("meta_analysis", "H2_logFC_concordance", "Fig2_H2_control_type_concordance"),
          ("clinical", "Forest_S_M_R_m1_vs_m3", "Fig3_forest_S_M_R"),
          ("sensitivity", "Sensitivity_M_forest", "Fig5_sensitivity_M"),
          ("orthogonal", "H3_IvyGAP_M_minus_R", "Fig6_H3_IvyGAP"),
          ("sc", "SC_S_detection_in_malignant_cells", "FigS1_singlecell_detection"),
          ("validation", "Scores_vs_neuron_identity", "FigS2_scores_vs_neuron_identity"),
          ("enrich", "ORA_GOBP_meta", "FigS3_ORA_GOBP"),
          ("meta_analysis", "Meta_volcano", "FigS4_meta_volcano"),
          ("network", "WGCNA_module_trait_heatmap", "FigS5_WGCNA")]
for folder, stem, new in copies:
    for ext in ("png", "pdf", "svg"):
        src = P(folder, mkdir=False) / f"{stem}.{ext}"
        if src.exists():
            shutil.copy2(src, FIG / f"{new}.{ext}")
    if not (P(folder, mkdir=False) / f"{stem}.png").exists():
        print("not available (step skipped or not run):", folder, stem)

# ---- Fig 1: cohort flow -----------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 3.2))
labels = [f"{r.cohort}\n(n={r.n}, deaths={r.deaths})" for r in t1.itertuples()] + \
         [f"{r.cohort}\n(GBM {r.n_gbm} vs control {r.n_control})" for r in de_sum.itertuples()]
colors = ["#1f3a93"] * len(t1) + ["#0f9d94"] * len(de_sum)
ax.barh(range(len(labels)), [r.n for r in t1.itertuples()] + list(de_sum.n_gbm + de_sum.n_control), color=colors)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("samples analysed"); ax.set_title("Cohorts: survival (blue, H1) and tumour-vs-control (green, H2)", fontsize=10)
savefig(fig, "Fig1_cohorts", folder="figures"); plt.close(fig)

In [ ]:
# ---- results digest and slide outline (plain text, numbers filled in) --------------------------------------
h1 = summary.get("H1") or {}; h2 = summary.get("H2") or {}
sc = summary.get("single_cell") or {}; sens = summary.get("sensitivity") or {}
S_info = manifest.get("gene_sets", {}).get("S", {})
sw = manifest.get("software", {})


def hr_line(model, pred):
    x = cm[(cm.model == model) & (cm.predictor == pred)]
    if not len(x):
        return "not evaluable"
    x = x.iloc[0]
    return f"HR {x.HR_fixed:.2f} (95% CI {x.HR_fixed_lo:.2f}-{x.HR_fixed_hi:.2f}), p={x.p_fixed:.3g}, I2={x.I2:.0f}%"


digest = f"""# GBM multi-cohort study: results digest
Generated {now_utc()} from the pipeline outputs. Every number below comes from a saved file.

## Population and data
- Adult, primary, IDH-wildtype GBM. Survival cohorts: {', '.join(f"{r.cohort} n={r.n} ({r.deaths} deaths)" for r in t1.itertuples())}.
- Tumour-vs-control cohorts: {', '.join(f"{r.cohort} ({r.n_gbm} GBM vs {r.n_control} controls)" for r in de_sum.itertuples())}.
- Synaptic set S: {S_info.get('source', 'see manifest')} ({S_info.get('n_genes', '?')} genes).
- Single cell (GSE131928): M = {sc.get('n_M', '?')} genes expressed in malignant cells, R = {sc.get('n_R', '?')} remaining genes, from {sc.get('eligible_units', '?')} patient units.

## H1 (primary): verdict = {str(h1.get('verdict', 'not run')).upper()}
- S, m1 (age, sex): {hr_line('m1', 'S')}
- S, m3 (+ purity proxies + neuron score): {hr_line('m3', 'S')}
- M, m3: {hr_line('m3', 'M')}
- R, m3: {hr_line('m3', 'R')}
- Reasons: {' | '.join(h1.get('reasons', []))}

## H2 (secondary): verdict = {str(h2.get('verdict', 'not run')).upper()}
- Lowest S-gene Spearman rho across control pairs: {h2.get('S_gene_min_spearman', float('nan')):.2f} (support if < 0.7).
- Marker sets enriched among DEGs: {len(h2.get('marker_sets_enriched', []))}.

## H3 (exploratory): {summary.get('H3')}

## Robustness
- M criterion met in {sens.get('n_M_criterion_met', '?')} of {sens.get('n_variants_M', '?')} sensitivity variants. Failing: {sens.get('variants_failing', [])}.
- Candidate tiers (per-gene, descriptive): {summary.get('tier_counts')}.

## Limitations (state these in the paper)
{chr(10).join('- ' + x for x in summary.get('limitations', []))}

## Methods summary
Expression data were processed per platform (Affymetrix arrays: log2 and quantile normalisation, probe-to-gene mapping;
TCGA/GTEx: Toil expected counts, TMM, voom; CGGA: log2 FPKM). Differential expression used limma (arrays, intensity-trend
robust eBayes) and limma-voom (TCGA vs GTEx, sex covariate); cohort log fold changes were pooled by DerSimonian-Laird
random-effects meta-analysis. Gene-set scores were mean z-scores within each cohort. Malignant cells in GSE131928 were
called with a CNV-style score; M genes were S genes detected in >= {CFG['sc']['detect_fraction_in_patient']:.0%} of malignant
cells in >= {CFG['sc']['min_patient_fraction']:.0%} of patients, frozen with checksums before survival analysis. Cox models
(per SD, no cut-points) were adjusted for age, sex, two marker-based purity proxies and a normal-neuron score, combined by
fixed- and random-effects meta-analysis. Software: Python {sw.get('python_and_system', {}).get('python', '?')}, R packages in run_manifest.json.
"""
(MS / "results_digest.md").write_text(digest)

slides = f"""# Slide outline (fill in design later)
1. Title: Separating malignant neural-like signal from normal-brain admixture in IDH-wildtype GBM
2. Background: neural/synaptic programs are prognostic in GBM, but bulk tissue also contains normal neurons
3. Question and hypotheses H1-H3 (pre-specified criteria)
4. Data: Fig1 cohorts - {', '.join(f"{r.cohort} n={r.n}" for r in t1.itertuples())}
5. Methods: S (SynGO) -> single cell M/R split -> scores -> Cox m1 vs m3 -> meta-analysis
6. H2 result: control type changes DEGs (Fig2) - verdict {str(h2.get('verdict', '?')).upper()}
7. Single cell: M = {sc.get('n_M', '?')}, R = {sc.get('n_R', '?')} (FigS1)
8. H1 result: Fig3 forest - M m3 {hr_line('m3', 'M')}; verdict {str(h1.get('verdict', '?')).upper()}
9. Survival display: Fig4 KM (median split, display only)
10. Robustness: Fig5 - M criterion met in {sens.get('n_M_criterion_met', '?')}/{sens.get('n_variants_M', '?')} variants
11. Exploratory H3 spatial (Fig6): {summary.get('H3')}
12. Limitations and next step (perturb top M genes in neuron-glioma co-culture)
"""
(MED / "slide_outline.md").write_text(slides)
print(digest)

In [ ]:
bundle = FIN / "GBM_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(FIG.glob("*.png")) + sorted(FIG.glob("*.pdf")):
        z.write(f, f"figures/{f.name}")
    for f in sorted(FIN.glob("*.csv")) + [FIN / "All_tables.xlsx", FIN / "results_summary.json"]:
        if f.exists():
            z.write(f, f"tables/{f.name}")
    for f in [MS / "results_digest.md", MED / "slide_outline.md", P("logs", "run_manifest.json"), P("config", "frozen_definitions.json")]:
        if f.exists():
            z.write(f, f"text/{f.name}")
print("Bundle:", bundle, f"({bundle.stat().st_size / 1e6:.1f} MB)")
manifest_update("final", "bundle", str(bundle))
print("\nNB15 complete. Share GBM_results_bundle.zip to build the presentation and the manuscript.")